# Chart of the Day — Iceland EU accession referendum

**29 August 2026.** Votes for and against restarting EU accession negotiations, nationally and by constituency, as a share of valid votes.

Iceland voted against reopening EU accession talks. The two Reykjavík constituencies were the only ones to back restarting negotiations. The Southwest constituency, covering the suburban municipalities around the capital, voted no by 53.0% to 47.0%.

### Source

RÚV results page, all votes counted, last updated 30 August 2026 at 09:28. https://www.ruv.is/english/2026-08-29-iceland-rejects-eu-accession-talks-485377

National result: 118,040 no (52.8%), 105,339 yes (47.2%), margin 12,701. Blank and invalid ballots 1,652 (0.7% of votes cast). Turnout 82.5% on 225,031 votes cast, above the 80.2% at the 2024 Althing election.

Constituency shares are RÚV's published figures to one decimal place. Vísir reported Southwest more precisely as 46.95% yes to 53.05% no; RÚV's 47.0 / 53.0 is used here so every row traces to one source.

**Caveat:** RÚV publishes the constituency returns, not the certified result. Landskjörstjórn declares the official result after ruling on disputed ballots and publishes it in Lögbirtingablaðið. Swap the source line once that lands.

**Not used:** Euronews describes the 82.5% turnout as a referendum record. Iceland's 1944 independence vote drew 98.4% and the 2011 Icesave vote 75.3%, so 'highest since 1944' would be defensible but 'record' is not. The claim is left out of the copy.

In [13]:
"""
Chart of the Day — Iceland EU accession referendum, 29 August 2026.

Votes for and against restarting EU accession negotiations, by constituency,
as a share of valid votes.

Data: final counts from the six constituency returning officers (yfirkjörstjórnir),
as published by RÚV on 30 August 2026 with all votes counted. Official results pending
declaration by Landskjörstjórn.
"""

import os
import json

import altair as alt
import pandas as pd

import eco_style
from eco_style import pallete

alt.theme.enable("report")

try:
    HERE = os.path.dirname(os.path.abspath(__file__))
except NameError:  # Positron / Jupyter
    HERE = os.getcwd()

## Data

Row order runs national result first, then a blank spacer row, then the six constituencies sorted by yes share. The spacer is kept in the y scale domain but carries no bars.

In [14]:
raw = pd.read_csv(os.path.join(HERE, "iceland_eu_referendum_2026.csv"))

seats = (
    raw[raw["level"] == "constituency"]
    .sort_values("yes_pct", ascending=False)
    .reset_index(drop=True)
)
national = raw[raw["level"] == "national"]

SPACER = " "  # blank row separating the national result from the constituencies

# Row order: the national result, a gap, then the six constituencies
ORDER = national["constituency"].tolist() + [SPACER] + seats["constituency"].tolist()

plot = pd.concat([national, seats], ignore_index=True)

plot["yes_label"] = plot["yes_pct"].map(lambda v: f"{v:.1f}%")
plot["no_label"] = plot["no_pct"].map(lambda v: f"{v:.1f}%")
plot["left"] = 0.0
plot["right"] = 100.0

## Encodings and chart

In [15]:
BLUE = pallete["nominal_1"]  # yes
NAVY = pallete["bar"]["accent_1"]  # no

SCALE = alt.Scale(domain=[0, 100], nice=False)

x_share = alt.X("yes_pct:Q", scale=SCALE, axis=None)
x_left = alt.X("left:Q", scale=SCALE, axis=None)
x_right = alt.X("right:Q", scale=SCALE, axis=None)

y_area = alt.Y(
    "constituency:N",
    sort=ORDER,
    scale=alt.Scale(domain=ORDER, paddingInner=0.42),
    axis=alt.Axis(
        title=None, labelFontSize=12, domain=False, ticks=False, labelPadding=10
    ),
)

# Yes segment, from 0 out to the yes share
yes_bar = alt.Chart(plot).mark_bar(color=BLUE).encode(x=x_share, x2="left:Q", y=y_area)

# No segment, from the yes share out to 100
no_bar = alt.Chart(plot).mark_bar(color=NAVY).encode(x=x_share, x2="right:Q", y=y_area)

yes_value = (
    alt.Chart(plot)
    .mark_text(align="left", dx=10, fontSize=12, fontWeight=500, color="white")
    .encode(x=x_left, y=y_area, text="yes_label:N")
)

no_value = (
    alt.Chart(plot)
    .mark_text(align="right", dx=-10, fontSize=12, fontWeight=500, color="white")
    .encode(x=x_right, y=y_area, text="no_label:N")
)

# 50% marker over the bars, so it is obvious which side crossed it
fifty = (
    alt.Chart(pd.DataFrame({"yes_pct": [50.0]}))
    .mark_rule(strokeWidth=1.2, color="white", opacity=0.9)
    .encode(x=x_share)
)

# Direct labels above the bars instead of a legend
head_yes = (
    alt.Chart(plot.head(1))
    .mark_text(align="left", fontSize=13, fontWeight=600, color=BLUE, baseline="bottom")
    .encode(x=x_left, y=alt.value(-10), text=alt.datum("Yes"))
)

head_no = (
    alt.Chart(plot.head(1))
    .mark_text(align="right", fontSize=13, fontWeight=600, color=NAVY, baseline="bottom")
    .encode(x=x_right, y=alt.value(-10), text=alt.datum("No"))
)

head_fifty = (
    alt.Chart(pd.DataFrame({"yes_pct": [50.0]}))
    .mark_text(
        align="center", fontSize=11, color="rgba(24, 42, 56, 0.6)", baseline="bottom"
    )
    .encode(x=x_share, y=alt.value(-10), text=alt.datum("50%"))
)

chart = (
    alt.layer(
        yes_bar,
        no_bar,
        fifty,
        yes_value,
        no_value,
        head_yes,
        head_no,
        head_fifty,
    )
    .properties(
        width=580,
        height=290,
        title=alt.TitleParams(
            text="Iceland says no to reopening EU talks",
            subtitle=[
                "Share of valid votes for and against restarting EU accession negotiations, nationally and by constituency.",
            ],
            fontSize=19,
            fontWeight=600,
            subtitleFontSize=13,
            subtitleColor="rgba(24, 42, 56, 0.75)",
            subtitleLineHeight=18,
            anchor="start",
            offset=32,
            dy=-6,
        ),
    )
)

note = (
    alt.Chart(plot.head(1))
    .mark_text(
        align="left",
        baseline="top",
        fontSize=10.5,
        lineBreak="\n",
        color="rgba(24, 42, 56, 0.6)",
        lineHeight=14,
        dy=18,
    )
    .encode(
        x=alt.value(0),
        y=alt.value("height"),
        text=alt.datum(
            "Source: RÚV, all votes counted, 30 August 2026. Official results pending declaration by Landskjörstjórn."
        ),
    )
)

final = (
    alt.layer(chart, note)
    .configure_view(stroke=None, strokeWidth=0)
    .configure_axis(grid=False)
    .properties(padding={"left": 6, "right": 18, "top": 6, "bottom": 10})
)

## Preview

In [16]:
final

alt.LayerChart(...)

## Save outputs

In [17]:
final.save(os.path.join(HERE, "chart_iceland_referendum.png"), scale_factor=3)
final.save(os.path.join(HERE, "chart_iceland_referendum.svg"))

with open(os.path.join(HERE, "chart_iceland_referendum.json"), "w") as f:
    json.dump(final.to_dict(), f, indent=2, ensure_ascii=False)

print("written to", HERE)

written to /Users/alonso/Desktop/LSE/GROWTH LAB/ChartofthedayRepo/iceland-ref


# Chart of the Day — Iceland EU accession referendum
 
**Chart:** `chart_iceland_referendum.png`
**Referendum date:** 29 August 2026
**Source:** RÚV, all votes counted, 30 August 2026
 
---
 
## X — 278 characters
 
```text
Iceland rejected the proposal to reopen EU accession negotiations. The no side won 52.8% to 47.2%, on a turnout of 82.5%.
 
Reykjavík diverged from the rest of the country. Its two constituencies supported reopening talks, by 57.5% and 54.5%. All others voted no.
 
#ChartOfTheDay
```
 
Limit 280. Two characters spare.
 
---
 
## Bluesky — 297 characters
 
```text
Iceland rejected reopening EU accession negotiations. The no side won 52.8% to 47.2%.
 
The ballot did not ask voters to approve EU membership, only whether to resume talks.
 
Reykjavík differed from the rest of the country, voting yes by 57.5% and 54.5%. All other regions voted no.
 
#ChartOfTheDay
```
 
Limit 300. Three characters spare.
 
---
 
## LinkedIn — 137 words
 
```text
Iceland rejected the proposal to reopen European Union accession negotiations. The no side won 52.8% to 47.2%, on a turnout of 82.5%.
 
Reykjavík's two constituencies supported reopening negotiations, by 57.5% and 54.5%. Every constituency outside the capital voted no, including 53.0% in the suburban Southwest and over 60% in the South, Northeast, and Northwest.
 
The referendum question was narrow. The ballot asked only whether to resume accession negotiations, which Iceland suspended more than a decade ago. Any eventual agreement would have required a second referendum.
 
Fisheries policy was the central issue in the campaign. Supporters of reopening talks argued that EU membership would reduce borrowing costs and stabilise the króna. Opponents said EU policies would threaten Icelandic control over fishing grounds. The prime minister says the issue is now closed for the duration of her government.
 
#ChartOfTheDay
```
 
No hard limit. LinkedIn truncates at roughly 210 characters, so the opening line carries the finding.
 
---
 
## Thread reply (X)
 
```text
🔗 Visit our Data Hub to explore and create your own charts. https://economicsobservatory.com/explore
```
 
---